# Moon + Mars landform classifier

One model for both worlds: 15 Mars landforms (DoMars16k) and up to 4 Moon classes
(LROCNet, plus rockfalls if that dataset can be downloaded). The website uses it to tell whether an
orbital image shows the Moon or Mars, which landform it is, and which places on Earth match.

**Before you run (Kaggle):**
1. Right panel → **Session options** → **Accelerator: GPU T4 x2** (or P100).
2. Right panel → **Internet: On**.
3. Top menu → **Run All**. Takes about 30–50 minutes.

**Data (all open):**
- DoMars16k: 16,150 Mars CTX images, 15 landforms, CC-BY-4.0. Wilhelm et al. 2020, doi:10.5281/zenodo.4291940
- LROCNet: ~5,000 LRO NAC Moon images, 3 classes (fresh crater, old crater, none). Dunkel/JPL 2022, doi:10.5281/zenodo.7041842
- Optional: lunar rockfalls (RMaM-2020), CC-BY-4.0. Bickel, Mandrake & Doran 2021, doi:10.3389/frsen.2021.640034

In [ ]:
import hashlib, json, random, shutil, time, urllib.request, zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms

try:
    import timm
except ImportError:
    %pip install -q timm
    import timm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}

def is_image(path):
    # Zips made on a Mac carry hidden '._name.jpg' files (in __MACOSX) that are not images.
    return path.suffix.lower() in IMAGE_EXT and not path.name.startswith('._') and '__MACOSX' not in path.parts
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(no GPU: turn it on in Session options)')

## 1. Mars: DoMars16k (128 MB), checked with its published MD5

In [ ]:
URL = 'https://zenodo.org/api/records/4291940/files/data.zip/content'
MD5 = '6457e1fa5a685af059ed939ee2ec3eea'
zip_path, mars_root = WORK / 'domars16k.zip', WORK / 'domars16k'

if not mars_root.exists():
    urllib.request.urlretrieve(URL, zip_path)
    digest = hashlib.md5(zip_path.read_bytes()).hexdigest()
    assert digest == MD5, f'Download corrupted: md5 {digest}'
    zipfile.ZipFile(zip_path).extractall(mars_root)
    zip_path.unlink()

def find_split(name):
    hits = [p for p in mars_root.rglob(name) if p.is_dir() and any(p.iterdir())]
    assert hits, f'No "{name}" folder found under {mars_root}'
    return hits[0]

MARS_SPLITS = {s: find_split(s) for s in ('train', 'val', 'test')}
for s, p in MARS_SPLITS.items():
    print('Mars', s, sum(1 for f in p.rglob('*') if is_image(f)), 'images')

## 2. Moon: LROCNet, checked with Zenodo's MD5 for every file
Folders `ejecta` = fresh crater, `oldcrater` = old crater, `none` = no distinct feature.

In [ ]:
lroc_root = WORK / 'lrocnet'
if not lroc_root.exists():
    record = json.load(urllib.request.urlopen('https://zenodo.org/api/records/7041842'))
    for f in record['files']:
        if not f['key'].endswith('.zip'):
            continue  # skip the labelling-guide slides
        dest = WORK / f['key']
        urllib.request.urlretrieve(f['links']['self'], dest)
        algo, want = f['checksum'].split(':')
        assert algo == 'md5' and hashlib.md5(dest.read_bytes()).hexdigest() == want, f'Download corrupted: {f["key"]}'
        zipfile.ZipFile(dest).extractall(lroc_root / dest.stem)
        dest.unlink()

def lroc_split(zip_stem):
    return 'val' if zip_stem == 'val' else 'test' if zip_stem == 'test' else 'train'

def lroc_class(path):
    text = str(path).lower()
    for key, cls in (('ejecta', 'moon_fresh'), ('oldcrater', 'moon_old'), ('none', 'moon_none')):
        if key in text:
            return cls
    return None

MOON = {s: [] for s in ('train', 'val', 'test')}  # (image path, class)
for img in lroc_root.rglob('*'):
    if is_image(img) and (cls := lroc_class(img.relative_to(lroc_root))):
        MOON[lroc_split(img.relative_to(lroc_root).parts[0])].append((img, cls))
for s, items in MOON.items():
    counts = {c: sum(1 for _, k in items if k == c) for c in sorted({k for _, k in items})}
    print('Moon', s, len(items), 'images', counts)
assert all(MOON.values()), 'LROCNet layout not as expected: check the folders printed above'

## 3. Moon rockfalls (optional)
Tries to download the lunar rockfall tiles (RMaM-2020). Tiles that contain rockfalls become the class
`moon_rockfall`. If the site is unreachable or the layout is unexpected, this step is skipped and the
model trains on the three LROCNet classes.

In [ ]:
ROCKFALL_DOI = 'doi:10.17617/3.7BXEVC'
rock_root, rockfall = WORK / 'rockfall', []
try:
    if not rock_root.exists():
        api = 'https://edmond.mpg.de/api'
        info = json.load(urllib.request.urlopen(f'{api}/datasets/:persistentId/?persistentId={ROCKFALL_DOI}', timeout=60))
        files = info['data']['latestVersion']['files']
        total_gb = sum(f['dataFile'].get('filesize', 0) for f in files) / 1e9
        print(f'{len(files)} files, {total_gb:.2f} GB')
        assert total_gb < 8, 'Too large for Kaggle working space'
        rock_root.mkdir()
        for f in files:
            df = f['dataFile']
            dest = rock_root / df['filename']
            urllib.request.urlretrieve(f'{api}/access/datafile/{df["id"]}', dest)
            if df.get('md5'):
                assert hashlib.md5(dest.read_bytes()).hexdigest() == df['md5'], f'Download corrupted: {dest.name}'
            if dest.suffix == '.zip':
                zipfile.ZipFile(dest).extractall(rock_root / dest.stem); dest.unlink()
    for img in rock_root.rglob('*'):
        text = str(img.relative_to(rock_root)).lower()
        is_moon = any(k in text for k in ('moon', 'lunar', 'lro', 'nac'))
        is_negative = any(k in text for k in ('neg', 'empty', 'background'))
        if is_image(img) and is_moon and not is_negative:
            rockfall.append(img)
    print('Lunar rockfall tiles found:', len(rockfall))
    if len(rockfall) < 100:
        print('Layout not recognised. Top-level contents:', sorted(p.name for p in rock_root.iterdir())[:30])
        rockfall = []
except Exception as error:
    print('Rockfall data skipped:', error)
    rockfall = []

if rockfall:
    rng = random.Random(SEED); rng.shuffle(rockfall)
    n_test, n_val = len(rockfall) // 10, len(rockfall) // 10
    MOON['test'] += [(p, 'moon_rockfall') for p in rockfall[:n_test]]
    MOON['val'] += [(p, 'moon_rockfall') for p in rockfall[n_test:n_test + n_val]]
    MOON['train'] += [(p, 'moon_rockfall') for p in rockfall[n_test + n_val:]]

## 4. One combined dataset (links, no copies)

In [ ]:
combined = WORK / 'combined'
shutil.rmtree(combined, ignore_errors=True)

from PIL import Image

counter, broken = {}, []
def add(src, split, cls):
    try:
        with Image.open(src) as image:
            image.verify()  # catches truncated or non-image files before training does
    except Exception:
        broken.append(src)
        return
    folder = combined / split / cls
    folder.mkdir(parents=True, exist_ok=True)
    n = counter[folder] = counter.get(folder, 0) + 1
    (folder / f'{n:05d}{src.suffix.lower()}').symlink_to(src.resolve())

for split, root in MARS_SPLITS.items():
    for class_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        for img in sorted(class_dir.iterdir()):
            if is_image(img):
                add(img, split, class_dir.name)

# "none" is most of LROCNet: cap it in training so it does not drown the other classes.
rng = random.Random(SEED)
none_train = sorted(img for img, cls in MOON['train'] if cls == 'moon_none')
rng.shuffle(none_train)
kept_none = set(none_train[:1100])
for split, items in MOON.items():
    for img, cls in items:
        if split == 'train' and cls == 'moon_none' and img not in kept_none:
            continue
        add(img, split, cls)

print('Skipped unreadable files:', len(broken), [str(p) for p in broken[:5]])
for split in ('train', 'val', 'test'):
    print(split, {p.name: len(list(p.iterdir())) for p in sorted((combined / split).iterdir())})

## 5. Data pipeline
Orbital images have no "up", so 90° rotations and flips are valid. Classes are sampled evenly.

In [ ]:
CLASS_NAMES = {
    'aec': 'Aeolian Curved', 'ael': 'Aeolian Straight', 'cli': 'Cliff', 'rid': 'Ridge',
    'fsf': 'Channel', 'sfe': 'Mounds', 'fsg': 'Gullies', 'fse': 'Slope Streaks',
    'fss': 'Mass Wasting', 'cra': 'Crater', 'sfx': 'Crater Field', 'mix': 'Mixed Terrain',
    'rou': 'Rough Terrain', 'smo': 'Smooth Terrain', 'tex': 'Textured Terrain',
    'moon_fresh': 'Moon: Fresh Crater', 'moon_old': 'Moon: Old Crater',
    'moon_none': 'Moon: Plain Surface', 'moon_rockfall': 'Moon: Rockfall',
}
body_of = lambda cls: 'Moon' if cls.startswith('moon_') else 'Mars'
IMG = 224
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class RandomRot90:
    def __call__(self, x):
        return torch.rot90(x, random.randint(0, 3), dims=(1, 2))

train_tf = transforms.Compose([
    transforms.Grayscale(3),
    transforms.RandomResizedCrop(IMG, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(), RandomRot90(), transforms.Normalize(MEAN, STD),
])
# Same as the website: centre square crop, resize to 224.
eval_tf = transforms.Compose([
    transforms.Grayscale(3), transforms.Resize(IMG), transforms.CenterCrop(IMG),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(combined / 'train', train_tf)
val_ds = datasets.ImageFolder(combined / 'val', eval_tf)
test_ds = datasets.ImageFolder(combined / 'test', eval_tf)
assert train_ds.classes == val_ds.classes == test_ds.classes
CLASSES = train_ds.classes
print(len(CLASSES), 'classes:', [CLASS_NAMES.get(c, c) for c in CLASSES])

counts = np.bincount(train_ds.targets, minlength=len(CLASSES))
sampler = WeightedRandomSampler([1.0 / counts[t] for t in train_ds.targets], num_samples=len(train_ds), replacement=True)
train_dl = DataLoader(train_ds, batch_size=64, sampler=sampler, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=64, num_workers=2, pin_memory=True)
test_dl = DataLoader(test_ds, batch_size=64, num_workers=2, pin_memory=True)

## 6. Fine-tune ConvNeXt-Nano (pretrained on ImageNet)

In [ ]:
EPOCHS = 14
MODEL_NAME = 'convnext_nano.in12k_ft_in1k'
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(CLASSES)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-4, total_steps=EPOCHS * len(train_dl), pct_start=0.1)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler(enabled=DEVICE == 'cuda')

@torch.no_grad()
def predict(dl, tta=False):
    model.eval()
    probs, labels = [], []
    for x, y in dl:
        x = x.to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
            views = [x] if not tta else [torch.rot90(v, k, dims=(2, 3)) for v in (x, x.flip(3)) for k in range(4)]
            p = torch.stack([model(v).float().softmax(1) for v in views]).mean(0)
        probs.append(p.cpu()); labels.append(y)
    return torch.cat(probs), torch.cat(labels)

def balanced_acc(probs, labels):  # mean per-class recall: fair when classes differ in size
    preds = probs.argmax(1)
    return float(np.mean([(preds[labels == c] == c).float().mean() for c in labels.unique()]))

best_acc, best_path = 0.0, WORK / 'planet_landforms_best.pt'
for epoch in range(1, EPOCHS + 1):
    model.train(); start, total_loss = time.time(), 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        total_loss += loss.item() * len(y)
    acc = balanced_acc(*predict(val_dl))
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), best_path)
    print(f'epoch {epoch:2d}  loss {total_loss / len(train_ds):.3f}  val balanced acc {acc:.2%}  best {best_acc:.2%}  ({time.time() - start:.0f}s)')

## 7. Test set (never seen in training): Mars, Moon and "which world" accuracy

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(best_path, map_location=DEVICE))
probs, labels = predict(test_dl, tta=True)
preds = probs.argmax(1)
names = [CLASS_NAMES.get(c, c) for c in CLASSES]
is_moon = torch.tensor([body_of(CLASSES[i]) == 'Moon' for i in labels])
pred_moon = torch.tensor([body_of(CLASSES[i]) == 'Moon' for i in preds])
correct = preds == labels
test_acc = correct.float().mean().item()
mars_acc = correct[~is_moon].float().mean().item()
moon_acc = correct[is_moon].float().mean().item()
body_acc = (is_moon == pred_moon).float().mean().item()
print(f'TEST ACCURACY  all {test_acc:.2%} | Mars {mars_acc:.2%} | Moon {moon_acc:.2%} | Moon-vs-Mars {body_acc:.2%}  ({len(labels)} images)\n')
print(classification_report(labels, preds, target_names=names, digits=3, zero_division=0))

cm = confusion_matrix(labels, preds, normalize='true')
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(names)), names, rotation=90); ax.set_yticks(range(len(names)), names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'Test accuracy {test_acc:.1%}')
plt.tight_layout(); plt.savefig(WORK / 'confusion_matrix.png', dpi=120); plt.show()

## 8. Save for the website
Download from the **Output** panel: `planet_landforms.int8.onnx` (the website model), `labels.json`,
`confusion_matrix.png`. The website keeps using the current Mars model until this one is wired in.

In [ ]:
try:
    import onnx, onnxscript, onnxruntime  # not all preinstalled on Kaggle
except ImportError:
    %pip install -q onnx onnxscript onnxruntime
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

fp32_path, int8_path = WORK / 'planet_landforms_fp32.onnx', WORK / 'planet_landforms.int8.onnx'
model.eval().cpu()
onnx_args = dict(input_names=['image'], output_names=['logits'])
try:
    torch.onnx.export(model, (torch.randn(1, 3, IMG, IMG),), fp32_path, external_data=False, **onnx_args)
except TypeError:  # older PyTorch without the new exporter options
    torch.onnx.export(model, (torch.randn(1, 3, IMG, IMG),), fp32_path, opset_version=17, **onnx_args)

graph_model = onnx.load(fp32_path)
del graph_model.graph.value_info[:]  # cached shapes trip the quantizer's shape inference
onnx.save(graph_model, fp32_path)
quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QInt8, op_types_to_quantize=['Conv', 'MatMul', 'Gemm'])

session = ort.InferenceSession(str(int8_path), providers=['CPUExecutionProvider'])
int8_preds = np.array([session.run(None, {'image': x.unsqueeze(0).numpy()})[0].argmax() for x, _ in test_ds])
targets = np.array(test_ds.targets)
int8_acc = float(np.mean(int8_preds == targets))
size_mb = lambda p: p.stat().st_size / 1e6
print(f'fp32 ONNX {size_mb(fp32_path):.1f} MB | int8 ONNX {size_mb(int8_path):.1f} MB')
print(f'TEST ACCURACY  PyTorch: {test_acc:.2%}  int8 ONNX: {int8_acc:.2%}')

meta = {
    'model': MODEL_NAME, 'classes': CLASSES, 'names': names, 'bodies': [body_of(c) for c in CLASSES],
    'image_size': IMG, 'mean': MEAN, 'std': STD, 'grayscale': True,
    'test_accuracy': round(test_acc, 4), 'mars_accuracy': round(mars_acc, 4), 'moon_accuracy': round(moon_acc, 4),
    'moon_vs_mars_accuracy': round(body_acc, 4), 'int8_test_accuracy': round(int8_acc, 4), 'test_images': len(labels),
    'datasets': ['DoMars16k doi:10.5281/zenodo.4291940', 'LROCNet doi:10.5281/zenodo.7041842']
                + (['Lunar rockfalls doi:10.3389/frsen.2021.640034'] if rockfall else []),
}
(WORK / 'labels.json').write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))

## 9. Sample images for the website
One random test image per class. Download `samples.zip`, unzip it, and upload everything inside to `public/samples/`.

In [ ]:
samples_dir = WORK / 'samples'
shutil.rmtree(samples_dir, ignore_errors=True)
samples_dir.mkdir()
rng = random.Random(SEED)
index = []
for class_dir in sorted(p for p in (combined / 'test').iterdir() if p.is_dir()):
    pick = rng.choice(sorted(class_dir.iterdir()))
    name = f'{class_dir.name}{pick.suffix.lower()}'
    shutil.copy(pick.resolve(), samples_dir / name)
    index.append({'file': name, 'landform': class_dir.name})
(samples_dir / 'index.json').write_text(json.dumps(index, indent=2))
shutil.make_archive(str(WORK / 'samples'), 'zip', samples_dir)
print(len(index), 'samples:', [s['file'] for s in index])